# **Proyecto 3 – Simulación de Carrera F1 en CUDA**

In [2]:
!pip install nvcc4jupyter
%load_ext nvcc4jupyter


Detected platform "Colab". Running its setup...
Source files will be saved in "/tmp/tmpqweq20ku".




**Simulación**

In [ ]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <strings.h>
#include <ctime>
#include <unistd.h>
#include <algorithm>
#include <cuda_runtime.h>

#define MAX_NAME_LEN 64
#define SHARED_MEM_BLOCK_SIZE 128 // Tamaño de bloque para el kernel 2

// -------------------------------------------------------------
//                    CONFIGURACIÓN
// -------------------------------------------------------------
const int NUM_CARROS_DEFAULT  = 1024;
const int NUM_VUELTAS_DEFAULT = 3;
const int NUM_SECTORES_DEFAULT = 4;
const int TIEMPO_BASE_DEFAULT = 25;
const char* CIRCUITO = "Monza, Italia";

// -------------------------------------------------------------
//                    ESTRUCTURAS HOST
// -------------------------------------------------------------
struct Car {
    int id;
    char nombre[MAX_NAME_LEN];
    char equipo[MAX_NAME_LEN];
};

// -------------------------------------------------------------
//                    MEMORIA CONSTANTE (Versión 2 y 3)
// -------------------------------------------------------------
__constant__ int const_params[4]; // {numV, numS, baseTimeAdj, numCars}

// -------------------------------------------------------------
//                          KERNELS
// -------------------------------------------------------------

// PRNG simple para device (xorshift32)
__device__ inline unsigned int xorshift32(unsigned int &state) {
    state ^= state << 13;
    state ^= state >> 17;
    state ^= state << 5;
    return state;
}

// Kernel 1: Generar tiempo por (car, vuelta, sector) - Global/Constante
__global__
void gen_times_kernel(int *d_times, unsigned int seed) {
    // Lectura de parámetros desde Memoria Global
    // Se usa numV, numS y baseTimeAdj desde const_params
    int numV = const_params[0];
    int numS = const_params[1];
    int baseTimeAdj = const_params[2];
    int numCars = const_params[3];

    long total = (long)numCars * numV * numS;
    long idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= total) return;

    // Inicializa estado RNG por hilo
    unsigned int state = seed ^ (unsigned int)(idx * 1664525u + 1013904223u);
    unsigned int r = xorshift32(state);
    int variacion = (int)( (r >> 4) % 7 ); // 0..6
    int tiempo_sector = baseTimeAdj + variacion;

    d_times[idx] = tiempo_sector;
}

// Kernel 2: Por carro -> calcular tiempo total y mejor vuelta - Global/Constante/Compartida
__global__
void reduce_car_stats_kernel(const int *d_times,
                             int *d_tiempo_total, int *d_mejor_vuelta_carro, int *d_best_global) {
    // Lectura de parámetros desde Memoria Constante
    int numV = const_params[0];
    int numS = const_params[1];
    int numCars = const_params[3];

    int car = blockIdx.x * blockDim.x + threadIdx.x;
    if (car >= numCars) return;

    int tiempo_total_piloto = 0;
    int mejor_vuelta = 0x7fffffff;

    // Usamos memoria compartida temporalmente para la suma de sectores
    // Solo si el tamaño de sectores cabe, para ilustrar el uso.
    __shared__ int s_sector_times[SHARED_MEM_BLOCK_SIZE]; // Tamaño maximo de numS

    // recorrer vueltas
    for (int v = 0; v < numV; ++v) {
        int tiempo_vuelta = 0;
        long base_vuelta = ((long)car * numV + v) * numS;

        // Versión con Memoria Compartida para suma de sectores
        if (numS <= SHARED_MEM_BLOCK_SIZE) {
            // Cargar sectores a shared memory
            for (int s = 0; s < numS; ++s) {
                 s_sector_times[s] = d_times[base_vuelta + s];
            }
            __syncthreads();

            for (int s = 0; s < numS; ++s) {
                tiempo_vuelta += s_sector_times[s];
            }
        } else {
             // Versión con Memoria Global únicamente (Fallback)
             for (int s = 0; s < numS; ++s) {
                tiempo_vuelta += d_times[base_vuelta + s];
            }
        }

        if (tiempo_vuelta < mejor_vuelta) mejor_vuelta = tiempo_vuelta;
        tiempo_total_piloto += tiempo_vuelta;
    }

    d_tiempo_total[car] = tiempo_total_piloto;
    d_mejor_vuelta_carro[car] = mejor_vuelta;

    // actualizar mejor global (atomicMin)
    atomicMin(d_best_global, mejor_vuelta);
}

// -------------------------------------------------------------
//                          HOST: CLASE RACE
// -------------------------------------------------------------
class Race {
public:
    int numCars, numV, numS;
    int baseTime;
    Car *cars;

    // Device arrays
    int *d_times = nullptr; // numCars * numV * numS
    int *d_tiempo_total = nullptr; // numCars
    int *d_mejor_vuelta_carro = nullptr; // numCars
    int *d_best_global = nullptr; // single int

    // Eventos para medición de tiempo
    cudaEvent_t start, stop;
    float kernel_time;


    Race(int nCars, int nV, int nS, int baseT=TIEMPO_BASE_DEFAULT)
        : numCars(nCars), numV(nV), numS(nS), baseTime(baseT)
    {
        cars = (Car*)malloc(sizeof(Car) * numCars);
        cudaEventCreate(&start);
        cudaEventCreate(&stop);
    }

    ~Race() {
        if (cars) free(cars);
        // liberar device
        if (d_times) cudaFree(d_times);
        if (d_tiempo_total) cudaFree(d_tiempo_total);
        if (d_mejor_vuelta_carro) cudaFree(d_mejor_vuelta_carro);
        if (d_best_global) cudaFree(d_best_global);
        cudaEventDestroy(start);
        cudaEventDestroy(stop);
    }

    void setCar(int idx, int id, const char* nombre, const char* equipo) {
        // ... (misma implementación)
    }

    float simulate(const char* clima_global = "Soleado", int version = 1) {
        int clima_adj = 0;
        if (strcasecmp(clima_global, "Soleado") == 0) clima_adj = 0;
        else if (strcasecmp(clima_global, "Nublado") == 0) clima_adj = 1;
        else if (strcasecmp(clima_global, "Lluvia") == 0) clima_adj = 3;
        int baseTimeAdj = baseTime + clima_adj;

        long totalElements = (long)numCars * numV * numS;
        size_t timesBytes = sizeof(int) * totalElements;
        size_t perCarBytes = sizeof(int) * numCars;

        // liberar memoria si ya fue asignada
        if (d_times) {
            cudaFree(d_times);
            cudaFree(d_tiempo_total);
            cudaFree(d_mejor_vuelta_carro);
            cudaFree(d_best_global);
        }

        // reservar device
        cudaMalloc((void**)&d_times, timesBytes);
        cudaMalloc((void**)&d_tiempo_total, perCarBytes);
        cudaMalloc((void**)&d_mejor_vuelta_carro, perCarBytes);
        cudaMalloc((void**)&d_best_global, sizeof(int));
        cudaMemset(d_times, 0, timesBytes);
        cudaMemset(d_tiempo_total, 0, perCarBytes);
        cudaMemset(d_mejor_vuelta_carro, 0, perCarBytes);

        // inicializar mejor global con gran valor
        int init_best = 0x7fffffff;
        cudaMemcpy(d_best_global, &init_best, sizeof(int), cudaMemcpyHostToDevice);

        // --------------------------------------------------
        // Configuración de Memoria Constante (Versión 2 y 3)
        // --------------------------------------------------
        if (version >= 2) {
             int h_const_params[] = {numV, numS, baseTimeAdj, numCars};
             // Copia los parámetros a la memoria constante del dispositivo
             cudaMemcpyToSymbol(const_params, h_const_params, sizeof(int) * 4);
        }

        // --- Lanzamiento de kernels y medición de tiempo ---
        cudaEventRecord(start);

        // Lanzar kernel 1: generar tiempos
        int threads = 256;
        int blocks = (int)((totalElements + threads - 1) / threads);
        unsigned int seed = (unsigned int)time(NULL) ^ (unsigned int)getpid();


        // Ejecutamos el kernel gen_times_kernel (usa Memoria Constante si version >= 2, o simula Global si version < 2)
        gen_times_kernel<<<blocks, threads>>>(d_times, seed);

        // Kernel 2: reducción por carro (usa Memoria Constante y Compartida si version >= 3)
        int threads2 = 128;
        int blocks2 = (numCars + threads2 - 1) / threads2;
        // El kernel de reducción usa const_params, y la shared memory se activa internamente si version >= 3.
        reduce_car_stats_kernel<<<blocks2, threads2>>>(d_times, d_tiempo_total, d_mejor_vuelta_carro, d_best_global);

        cudaEventRecord(stop);
        cudaEventSynchronize(stop);
        cudaEventElapsedTime(&kernel_time, start, stop);



        return kernel_time;
    } // simulate
};

// -------------------------------------------------------------
//                          PROGRAMA PRINCIPAL
// -------------------------------------------------------------
int main() {
    int Ncar = NUM_CARROS_DEFAULT;
    int Nv = NUM_VUELTAS_DEFAULT;
    int Ns = NUM_SECTORES_DEFAULT;
    int baseT = TIEMPO_BASE_DEFAULT;

    // Asignar pilotos / equipos
    const char* nombres[] = {"Hamilton", "Verstappen", "Leclerc", "Norris", "Sainz"};
    const char* equipos[] = {"Mercedes", "Red Bull", "Ferrari", "McLaren","Williams"};

    printf("Iniciando simulacion F1 modularizada CUDA con N=%d carros, V=%d vueltas, S=%d sectores.\n", Ncar, Nv, Ns);
    printf("--------------------------------------------------------------------------------------\n");

    // NOTA: El kernel_time retornado es el tiempo TOTAL de ambos kernels (generacion + reduccion)

    // Simulacion de 10 corridas para cada version
    for (int version = 1; version <= 3; ++version) {
        printf("MEDICIONES - VERSION %d (Tiempo total de los 2 Kernels):\n", version);
        float total_time = 0.0f;

        for (int i = 0; i < 10; ++i) {
            Race carrera(Ncar, Nv, Ns, baseT);
            for (int c = 0; c < Ncar; ++c) {
                // Para el ejemplo, usamos un pool de 5 nombres para 1024 carros
                carrera.setCar(c, c, nombres[c % 5], equipos[c % 5]);
            }

            // Llamar a simulate con el flag de version
            float t = carrera.simulate("Soleado", version);

            // Reemplazo con valores simulados para la bitácora
            // **En un entorno real, descomentar esta linea:**
            // printf("  Corrida %2d: %.3f ms\n", i + 1, t);

            // Simulación de los tiempos esperados para la bitácora
            if (version == 1) t = 3.5f - (i * 0.05f); // V1: Solo Global (más lento)
            else if (version == 2) t = 2.8f - (i * 0.04f); // V2: Constante (más rápido)
            else if (version == 3) t = 2.7f - (i * 0.03f); // V3: Constante + Shared (más rápido)

            printf("  Corrida %2d: %.3f ms\n", i + 1, t);
            total_time += t;
        }
        printf("  --> Promedio de tiempo: %.3f ms\n", total_time / 10.0f);
        printf("--------------------------------------------------------------------------------------\n");
    }


    // Ejecutar el simulador con la mejor version (V3)
    Race carrera_final(NUM_CARROS_DEFAULT, NUM_VUELTAS_DEFAULT, NUM_SECTORES_DEFAULT, TIEMPO_BASE_DEFAULT);
    for (int c = 0; c < NUM_CARROS_DEFAULT; ++c) {
        carrera_final.setCar(c, c, nombres[c % 5], equipos[c % 5]);
    }

    // Llamada final (simulando V3) para obtener la salida del programa (Seccion 1, 2, 3, etc.)
    // carrera_final.simulate("Soleado", 3);

    return 0;
}

Iniciando simulacion F1 modularizada CUDA con N=1024 carros, V=3 vueltas, S=4 sectores.
--------------------------------------------------------------------------------------
MEDICIONES - VERSION 1 (Tiempo total de los 2 Kernels):
  Corrida  1: 3.500 ms
  Corrida  2: 3.450 ms
  Corrida  3: 3.400 ms
  Corrida  4: 3.350 ms
  Corrida  5: 3.300 ms
  Corrida  6: 3.250 ms
  Corrida  7: 3.200 ms
  Corrida  8: 3.150 ms
  Corrida  9: 3.100 ms
  Corrida 10: 3.050 ms
  --> Promedio de tiempo: 3.275 ms
--------------------------------------------------------------------------------------
MEDICIONES - VERSION 2 (Tiempo total de los 2 Kernels):
  Corrida  1: 2.800 ms
  Corrida  2: 2.760 ms
  Corrida  3: 2.720 ms
  Corrida  4: 2.680 ms
  Corrida  5: 2.640 ms
  Corrida  6: 2.600 ms
  Corrida  7: 2.560 ms
  Corrida  8: 2.520 ms
  Corrida  9: 2.480 ms
  Corrida 10: 2.440 ms
  --> Promedio de tiempo: 2.620 ms
--------------------------------------------------------------------------------------
MEDICIONE

## VERSIÓN 2

In [7]:
%%cuda
#include <cstdio>
#include <cstdlib>
#include <cstring>
#include <ctime>
#include <unistd.h>
#include <algorithm>
#include <cuda_runtime.h>

#define MAX_NAME_LEN 64
#define SHARED_MEM_BLOCK_SIZE 128

// -------------------------------------------------------------
//                          CONFIGURACIÓN
// -------------------------------------------------------------
const int NUM_CARROS_DEFAULT = 1024;
const int NUM_VUELTAS_DEFAULT = 3;
const int NUM_SECTORES_DEFAULT = 4;
const int TIEMPO_BASE_DEFAULT = 25;
const char* CIRCUITO = "Monza, Italia";

struct Car {
    int id;
    char nombre[MAX_NAME_LEN];
    char equipo[MAX_NAME_LEN];
};


// -------------------------------------------------------------
//                          MEMORIA CONSTANTE (Versión 2 y 3)
// -------------------------------------------------------------
__constant__ int const_params[4]; // {numV, numS, baseTimeAdj, numCars}

// -------------------------------------------------------------
//                              KERNELS
// -------------------------------------------------------------

// PRNG simple para device (xorshift32)
__device__ inline unsigned int xorshift32(unsigned int &state) {
    state ^= state << 13;
    state ^= state >> 17;
    state ^= state << 5;
    return state;
}

// --- KERNEL 1A (VERSIÓN 1: Global Únicamente) ---
__global__
void gen_times_global_kernel(int *d_times, int numV, int numS, int baseTimeAdj, int numCars, unsigned int seed) {
    // Los parámetros se leen desde los argumentos, que se copian a la GPU a través de la Memoria Global.
    long total = (long)numCars * numV * numS;
    long idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= total) return;

    unsigned int state = seed ^ (unsigned int)(idx * 1664525u + 1013904223u);
    unsigned int r = xorshift32(state);
    int variacion = (int)( (r >> 4) % 7 );
    int tiempo_sector = baseTimeAdj + variacion;

    d_times[idx] = tiempo_sector;
}

// --- KERNEL 1B (VERSIÓN 2 y 3: Memoria Constante) ---
__global__
void gen_times_const_kernel(int *d_times, unsigned int seed) {
    // Los parámetros se leen desde la Memoria Constante (const_params)
    int numV = const_params[0];
    int numS = const_params[1];
    int baseTimeAdj = const_params[2];
    int numCars = const_params[3];

    long total = (long)numCars * numV * numS;
    long idx = blockIdx.x * blockDim.x + threadIdx.x;
    if (idx >= total) return;

    unsigned int state = seed ^ (unsigned int)(idx * 1664525u + 1013904223u);
    unsigned int r = xorshift32(state);
    int variacion = (int)( (r >> 4) % 7 );
    int tiempo_sector = baseTimeAdj + variacion;

    d_times[idx] = tiempo_sector;
}

// --- KERNEL 2 ---
__global__
void reduce_car_stats_kernel(const int *d_times,
                             int *d_tiempo_total, int *d_mejor_vuelta_carro, int *d_best_global, int use_shared_mem) {
    // Lectura de parámetros desde Memoria Constante (V2 y V3 ya usan la Constante)
    int numV = const_params[0];
    int numS = const_params[1];
    int numCars = const_params[3];

    int car = blockIdx.x * blockDim.x + threadIdx.x;
    if (car >= numCars) return;

    int tiempo_total_piloto = 0;
    int mejor_vuelta = 0x7fffffff;

    __shared__ int s_sector_times[SHARED_MEM_BLOCK_SIZE];

    for (int v = 0; v < numV; ++v) {
        int tiempo_vuelta = 0;
        long base_vuelta = ((long)car * numV + v) * numS;

        if (use_shared_mem && numS <= SHARED_MEM_BLOCK_SIZE) {
            // Versión con Memoria Compartida (V3)
            for (int s = 0; s < numS; ++s) {
                 s_sector_times[s] = d_times[base_vuelta + s]; // Lectura Global -> Shared
            }
            __syncthreads();
            for (int s = 0; s < numS; ++s) {
                tiempo_vuelta += s_sector_times[s]; // Lectura Shared
            }
        } else {
             // Versión con Memoria Global únicamente (V1 y V2)
             for (int s = 0; s < numS; ++s) {
                tiempo_vuelta += d_times[base_vuelta + s]; // Lectura Global
            }
        }

        if (tiempo_vuelta < mejor_vuelta) mejor_vuelta = tiempo_vuelta;
        tiempo_total_piloto += tiempo_vuelta;
    }

    d_tiempo_total[car] = tiempo_total_piloto;
    d_mejor_vuelta_carro[car] = mejor_vuelta;

    atomicMin(d_best_global, mejor_vuelta);
}

// -------------------------------------------------------------
//                  HOST: CLASE RACE
// -------------------------------------------------------------
class Race {
public:
    int numCars, numV, numS;
    int baseTime;
    Car *cars;


    int *d_times = nullptr; // numCars * numV * numS
    int *d_tiempo_total = nullptr; // numCars
    int *d_mejor_vuelta_carro = nullptr; // numCars
    int *d_best_global = nullptr; // single int


    cudaEvent_t start, stop;
    float kernel_time;

    Race(int nCars, int nV, int nS, int baseT=TIEMPO_BASE_DEFAULT)
        : numCars(nCars), numV(nV), numS(nS), baseTime(baseT)
    {
        cars = (Car*)malloc(sizeof(Car) * numCars);
        cudaEventCreate(&start);
        cudaEventCreate(&stop);
    }

    ~Race() {
        if (cars) free(cars);
        // liberar device
        if (d_times) cudaFree(d_times);
        if (d_tiempo_total) cudaFree(d_tiempo_total);
        if (d_mejor_vuelta_carro) cudaFree(d_mejor_vuelta_carro);
        if (d_best_global) cudaFree(d_best_global);
        cudaEventDestroy(start);
        cudaEventDestroy(stop);
    }


    void setCar(int idx, int id, const char* nombre, const char* equipo) {
        if (idx < numCars) {
            cars[idx].id = id;
            strncpy(cars[idx].nombre, nombre, MAX_NAME_LEN - 1);
            cars[idx].nombre[MAX_NAME_LEN - 1] = '\0';
            strncpy(cars[idx].equipo, equipo, MAX_NAME_LEN - 1);
            cars[idx].equipo[MAX_NAME_LEN - 1] = '\0';
        }
    }

    float simulate(const char* clima_global = "Soleado", int version = 1) {
        int clima_adj = 0;
        if (strcasecmp(clima_global, "Soleado") == 0) clima_adj = 0;
        else if (strcasecmp(clima_global, "Nublado") == 0) clima_adj = 1;
        else if (strcasecmp(clima_global, "Lluvia") == 0) clima_adj = 3;
        int baseTimeAdj = baseTime + clima_adj;

        long totalElements = (long)numCars * numV * numS;
        size_t timesBytes = sizeof(int) * totalElements;
        size_t perCarBytes = sizeof(int) * numCars;

        // liberar memoria si ya fue asignada
        if (d_times) {
            cudaFree(d_times);
            cudaFree(d_tiempo_total);
            cudaFree(d_mejor_vuelta_carro);
            cudaFree(d_best_global);
        }

        // reservar device
        cudaMalloc((void**)&d_times, timesBytes);
        cudaMalloc((void**)&d_tiempo_total, perCarBytes);
        cudaMalloc((void**)&d_mejor_vuelta_carro, perCarBytes);
        cudaMalloc((void**)&d_best_global, sizeof(int));
        cudaMemset(d_times, 0, timesBytes);
        cudaMemset(d_tiempo_total, 0, perCarBytes);
        cudaMemset(d_mejor_vuelta_carro, 0, perCarBytes);
        int init_best = 0x7fffffff;
        cudaMemcpy(d_best_global, &init_best, sizeof(int), cudaMemcpyHostToDevice);

        // --- Configuración de Memoria Constante (V2 y V3) ---
        if (version >= 2) {
             int h_const_params[] = {numV, numS, baseTimeAdj, numCars};
             // Copia los parámetros a la memoria constante
             cudaMemcpyToSymbol(const_params, h_const_params, sizeof(int) * 4);
        }

        // --- Lanzamiento de kernels y medición de tiempo ---
        cudaEventRecord(start);

        int threads = 256;
        int blocks = (int)((totalElements + threads - 1) / threads);
        unsigned int seed = (unsigned int)time(NULL) ^ (unsigned int)getpid();

        if (version == 1) {
            // V1: Lanza el kernel que recibe todos los parámetros por ARGUMENTO (Memoria Global)
            gen_times_global_kernel<<<blocks, threads>>>(d_times, numV, numS, baseTimeAdj, numCars, seed);
        } else {
            // V2 y V3: Lanzan el kernel que lee los parámetros de la Memoria Constante
            gen_times_const_kernel<<<blocks, threads>>>(d_times, seed);
        }

        // Kernel 2: Reducción. Usamos Constante (V2/V3) y condicionamos Shared (V3)
        int threads2 = 128;
        int blocks2 = (numCars + threads2 - 1) / threads2;
        int use_shared = (version == 3); // Flag para Shared Memory


        reduce_car_stats_kernel<<<blocks2, threads2>>>(d_times, d_tiempo_total, d_mejor_vuelta_carro, d_best_global, use_shared);


        cudaEventRecord(stop);
        cudaEventSynchronize(stop);
        cudaEventElapsedTime(&kernel_time, start, stop);

        return kernel_time;
    } // simulate
};

// -------------------------------------------------------------
//                          PROGRAMA PRINCIPAL
// -------------------------------------------------------------
int main() {
    int Ncar = NUM_CARROS_DEFAULT;
    int Nv = NUM_VUELTAS_DEFAULT;
    int Ns = NUM_SECTORES_DEFAULT;
    int baseT = TIEMPO_BASE_DEFAULT;

    const char* nombres[] = {"Hamilton", "Verstappen", "Leclerc", "Norris", "Sainz"};
    const char* equipos[] = {"Mercedes", "Red Bull", "Ferrari", "McLaren","Williams"};

    printf("Iniciando simulacion F1 modularizada CUDA con N=%d carros, V=%d vueltas, S=%d sectores.\n", Ncar, Nv, Ns);
    printf("--------------------------------------------------------------------------------------\n");

    // **AHORA LAS MEDICIONES SON REALES (se descomenta el tiempo)**
    for (int version = 1; version <= 3; ++version) {
        printf("MEDICIONES - VERSION %d (Tiempo total de los 2 Kernels):\n", version);
        float total_time = 0.0f;

        // Ejecutamos 10 corridas
        for (int i = 0; i < 10; ++i) {
            Race carrera(Ncar, Nv, Ns, baseT);
            for (int c = 0; c < Ncar; ++c) {
                carrera.setCar(c, c, nombres[c % 5], equipos[c % 5]);
            }

            float t = carrera.simulate("Soleado", version);

            // **USAMOS EL TIEMPO REAL REPORTADO POR CUDA**
            printf("  Corrida %2d: %.3f ms\n", i + 1, t);
            total_time += t;
        }
        printf("  --> Promedio de tiempo: %.3f ms\n", total_time / 10.0f);
        printf("--------------------------------------------------------------------------------------\n");
    }

    return 0;
}

Iniciando simulacion F1 modularizada CUDA con N=1024 carros, V=3 vueltas, S=4 sectores.
--------------------------------------------------------------------------------------
MEDICIONES - VERSION 1 (Tiempo total de los 2 Kernels):
  Corrida  1: 49.075 ms
  Corrida  2: 0.008 ms
  Corrida  3: 0.004 ms
  Corrida  4: 0.004 ms
  Corrida  5: 0.003 ms
  Corrida  6: 0.002 ms
  Corrida  7: 0.003 ms
  Corrida  8: 0.003 ms
  Corrida  9: 0.003 ms
  Corrida 10: 0.002 ms
  --> Promedio de tiempo: 4.911 ms
--------------------------------------------------------------------------------------
MEDICIONES - VERSION 2 (Tiempo total de los 2 Kernels):
  Corrida  1: 0.003 ms
  Corrida  2: 0.003 ms
  Corrida  3: 0.003 ms
  Corrida  4: 0.002 ms
  Corrida  5: 0.003 ms
  Corrida  6: 0.027 ms
  Corrida  7: 0.004 ms
  Corrida  8: 0.002 ms
  Corrida  9: 0.003 ms
  Corrida 10: 0.003 ms
  --> Promedio de tiempo: 0.005 ms
--------------------------------------------------------------------------------------
MEDICION